In [1]:
import pandas as pd
weather_path = r"C:\Users\Admin\OneDrive\Desktop\FYP_Dataset\master_dataset_features.csv"
weather_df = pd.read_csv(weather_path)
weather_df['datetime'] = pd.to_datetime(weather_df['datetime'])

city_dummy_cols = [c for c in weather_df.columns if c.startswith('city_')]
weather_df['city_name'] = weather_df[city_dummy_cols].idxmax(axis=1).str.replace('city_', '', regex=False)

print(" WEATHER DATASET ")
print(f"Shape: {weather_df.shape}")
print(f"Date range: {weather_df['datetime'].min()} to {weather_df['datetime'].max()}")
print(f"Cities: {weather_df['city_name'].unique()}")

pollution_path = r"C:\Users\Admin\OneDrive\Desktop\FYP_Dataset\pollution_data_2021_2025.csv"
pollution_df = pd.read_csv(pollution_path)
pollution_df['datetime'] = pd.to_datetime(pollution_df['datetime'])

print("\n POLLUTION DATASET ")
print(f"Shape: {pollution_df.shape}")
print(f"Date range: {pollution_df['datetime'].min()} to {pollution_df['datetime'].max()}")
print(f"Cities: {pollution_df['city_name'].unique()}")

 WEATHER DATASET 
Shape: (37234, 26)
Date range: 2021-11-01 00:00:00 to 2025-02-28 23:00:00
Cities: ['Faisalabad' 'Islamabad' 'Lahore' 'Multan']

 POLLUTION DATASET 
Shape: (56884, 11)
Date range: 2021-01-01 00:00:00 to 2025-12-31 17:00:00
Cities: ['Lahore' 'Islamabad' 'Faisalabad' 'Multan']


In [3]:

merged_df = pd.merge(
    weather_df,
    pollution_df,
    on=['city_name', 'datetime'],
    how='left' 
)

print(f"Weather rows before merge: {len(weather_df)}")
print(f"Merged rows: {len(merged_df)}")

matched = merged_df['pm2_5'].notna().sum()
missing = merged_df['pm2_5'].isna().sum()

print(f"\nRows WITH pollution data matched: {matched}")
print(f"Rows MISSING pollution data (no exact timestamp match): {missing}")
print(f"Match rate: {matched / len(merged_df) * 100:.1f}%")

Weather rows before merge: 37234
Merged rows: 37275

Rows WITH pollution data matched: 36741
Rows MISSING pollution data (no exact timestamp match): 534
Match rate: 98.6%


In [5]:

duplicates = pollution_df.duplicated(subset=['city_name', 'datetime'], keep=False)
print(f"Duplicate pollution rows found: {duplicates.sum()}")

if duplicates.sum() > 0:
    print("\nExample duplicates:")
    print(pollution_df[duplicates].sort_values(['city_name', 'datetime']).head(10))


pollution_df_clean = pollution_df.drop_duplicates(subset=['city_name', 'datetime'], keep='first')
print(f"\nPollution rows before: {len(pollution_df)}")
print(f"Pollution rows after removing duplicates: {len(pollution_df_clean)}")

merged_df = pd.merge(
    weather_df,
    pollution_df_clean,
    on=['city_name', 'datetime'],
    how='left'
)

print(f"\nWeather rows: {len(weather_df)}")
print(f"Merged rows (should now match): {len(merged_df)}")

matched = merged_df['pm2_5'].notna().sum()
missing = merged_df['pm2_5'].isna().sum()
print(f"Rows WITH pollution matched: {matched}")
print(f"Rows MISSING pollution: {missing}")
print(f"Match rate: {matched / len(merged_df) * 100:.1f}%")

Duplicate pollution rows found: 112

Example duplicates:
        city_name   datetime  aqi       co    no    no2      o3    so2  \
30628  Faisalabad 2021-02-01    5  1869.20  2.04  20.05  181.68  29.56   
30629  Faisalabad 2021-02-01    5  1869.20  2.04  20.05  181.68  29.56   
29162  Faisalabad 2021-12-01    5  1161.58  1.54  14.57  171.66  15.50   
29163  Faisalabad 2021-12-01    5  1161.58  1.54  14.57  171.66  15.50   
29907  Faisalabad 2022-01-01    5  1708.98  5.53  32.22  127.32  16.93   
32696  Faisalabad 2022-01-01    5  1708.98  5.53  32.22  127.32  16.93   
33416  Faisalabad 2022-02-01    5  3417.97  3.74  40.78  208.85  21.93   
33417  Faisalabad 2022-02-01    5  3417.97  3.74  40.78  208.85  21.93   
32022  Faisalabad 2022-12-01    5  1602.17  5.14  34.27  135.90  23.13   
32023  Faisalabad 2022-12-01    5  1602.17  5.14  34.27  135.90  23.13   

        pm2_5    pm10    nh3  
30628  285.44  312.08  15.33  
30629  285.44  312.08  15.33  
29162  116.18  131.18   4.56  
2916

In [7]:

final_df = merged_df.dropna(subset=['pm2_5']).reset_index(drop=True)

print(f"Rows before dropping: {len(merged_df)}")
print(f"Rows after dropping missing pollution: {len(final_df)}")

output_path = r"C:\Users\Admin\OneDrive\Desktop\FYP_Dataset\master_dataset_with_pollution.csv"
final_df.to_csv(output_path, index=False)
print(f"\nSaved merged dataset to: {output_path}")

print("\nColumns in final dataset:")
print(final_df.columns.tolist())

Rows before dropping: 37234
Rows after dropping missing pollution: 36700

Saved merged dataset to: C:\Users\Admin\OneDrive\Desktop\FYP_Dataset\master_dataset_with_pollution.csv

Columns in final dataset:
['datetime', 'temp', 'humidity', 'dew', 'windspeed', 'windgust', 'winddir', 'sealevelpressure', 'cloudcover', 'visibility', 'precip', 'snow', 'snowdepth', 'month', 'hour', 'is_smog_prone_hour', 'dew_point_depression', 'city_Faisalabad', 'city_Islamabad', 'city_Lahore', 'city_Multan', 'visibility_prev_reading', 'time_gap_hours', 'winddir_sin', 'winddir_cos', 'city_name', 'aqi', 'co', 'no', 'no2', 'o3', 'so2', 'pm2_5', 'pm10', 'nh3']


In [9]:
import numpy as np
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, classification_report
import xgboost as xgb


final_df = final_df.sort_values('datetime').reset_index(drop=True)


exclude_cols = ['visibility', 'datetime', 'time_gap_hours', 'city_name']
feature_cols = [c for c in final_df.columns if c not in exclude_cols]

X = final_df[feature_cols]
y = final_df['visibility']

print(f"Features used ({len(feature_cols)}): {feature_cols}")


split_index = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

print(f"\nTraining rows: {len(X_train)}")
print(f"Testing rows: {len(X_test)}")

sample_weights = np.where(y_train < 0.5, 5, 1)

xgb_pollution_model = xgb.XGBRegressor(
    n_estimators=100, max_depth=6, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    random_state=42, n_jobs=-1
)

xgb_pollution_model.fit(X_train, y_train, sample_weight=sample_weights)
print("\nTraining complete!")

pred = xgb_pollution_model.predict(X_test)

r2 = r2_score(y_test, pred)
mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))

print(f"\n=== WITH POLLUTION — Regression ===")
print(f"R²   : {r2:.4f}")
print(f"MAE  : {mae:.4f} km")
print(f"RMSE : {rmse:.4f} km")

def classify_risk(v):
    if v < 0.5: return "CRITICAL"
    elif v < 2.0: return "HIGH"
    elif v < 5.0: return "MODERATE"
    elif v < 10.0: return "LOW"
    else: return "SAFE"

actual_risk = y_test.apply(classify_risk)
predicted_risk = pd.Series(pred, index=y_test.index).apply(classify_risk)

print(f"\n=== WITH POLLUTION — Risk Classification ===")
print(classification_report(actual_risk, predicted_risk,
                             labels=['CRITICAL','HIGH','MODERATE','LOW','SAFE']))

overall_accuracy = (actual_risk == predicted_risk).mean()
print(f"Overall classification accuracy: {overall_accuracy*100:.1f}%")

Features used (31): ['temp', 'humidity', 'dew', 'windspeed', 'windgust', 'winddir', 'sealevelpressure', 'cloudcover', 'precip', 'snow', 'snowdepth', 'month', 'hour', 'is_smog_prone_hour', 'dew_point_depression', 'city_Faisalabad', 'city_Islamabad', 'city_Lahore', 'city_Multan', 'visibility_prev_reading', 'winddir_sin', 'winddir_cos', 'aqi', 'co', 'no', 'no2', 'o3', 'so2', 'pm2_5', 'pm10', 'nh3']

Training rows: 29360
Testing rows: 7340

Training complete!

=== WITH POLLUTION — Regression ===
R²   : 0.9114
MAE  : 1.5471 km
RMSE : 2.7771 km

=== WITH POLLUTION — Risk Classification ===
              precision    recall  f1-score   support

    CRITICAL       0.64      0.60      0.62       205
        HIGH       0.66      0.54      0.59       816
    MODERATE       0.86      0.77      0.81      3183
         LOW       0.41      0.68      0.51       737
        SAFE       0.91      0.92      0.92      2399

    accuracy                           0.78      7340
   macro avg       0.70      

In [11]:
import joblib

model_path = r"C:\Users\Admin\OneDrive\Desktop\FYP_Dataset\xgb_model_with_pollution_final.pkl"
joblib.dump(xgb_pollution_model, model_path)
print(f"Model saved to: {model_path}")


test_export = final_df.iloc[X_test.index].copy()
test_export['predicted_visibility'] = pred
test_export['actual_risk'] = actual_risk.values
test_export['predicted_risk'] = predicted_risk.values

test_export_path = r"C:\Users\Admin\OneDrive\Desktop\FYP_Dataset\test_set_with_pollution.csv"
test_export.to_csv(test_export_path, index=False)
print(f"Test set with predictions saved to: {test_export_path}")

Model saved to: C:\Users\Admin\OneDrive\Desktop\FYP_Dataset\xgb_model_with_pollution_final.pkl
Test set with predictions saved to: C:\Users\Admin\OneDrive\Desktop\FYP_Dataset\test_set_with_pollution.csv


In [22]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
import time

def evaluate_model(name, y_test, pred):
    r2 = r2_score(y_test, pred)
    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))

    actual_risk = y_test.apply(classify_risk)
    predicted_risk = pd.Series(pred, index=y_test.index).apply(classify_risk)
    accuracy = (actual_risk == predicted_risk).mean()

    report = classification_report(actual_risk, predicted_risk,
                                    labels=['CRITICAL','HIGH','MODERATE','LOW','SAFE'],
                                    output_dict=True)
    critical_recall = report['CRITICAL']['recall']

    print(f"\n=== {name} ===")
    print(f"R²: {r2:.4f} | MAE: {mae:.4f} | RMSE: {rmse:.4f}")
    print(f"Classification Accuracy: {accuracy*100:.1f}% | CRITICAL Recall: {critical_recall*100:.1f}%")

    return {'model': name, 'r2': r2, 'mae': mae, 'rmse': rmse,
            'accuracy': accuracy, 'critical_recall': critical_recall}

results = []

# ============================================================
# XGBoost (already trained above — just re-evaluate for the table)
# ============================================================
xgb_pred = xgb_pollution_model.predict(X_test)
results.append(evaluate_model("XGBoost", y_test, xgb_pred))

# ============================================================
# Random Forest
# ============================================================
print("\nTraining Random Forest...")
start = time.time()

rf_model = RandomForestRegressor(
    n_estimators=100, max_depth=None,
    random_state=42, n_jobs=-1
)
rf_model.fit(X_train, y_train, sample_weight=sample_weights)
rf_pred = rf_model.predict(X_test)

print(f"Random Forest trained in {time.time()-start:.1f} seconds")
results.append(evaluate_model("Random Forest", y_test, rf_pred))






=== XGBoost ===
R²: 0.9114 | MAE: 1.5471 | RMSE: 2.7771
Classification Accuracy: 77.8% | CRITICAL Recall: 60.0%

Training Random Forest...
Random Forest trained in 62.8 seconds

=== Random Forest ===
R²: 0.8703 | MAE: 1.6205 | RMSE: 3.3599
Classification Accuracy: 78.5% | CRITICAL Recall: 52.7%


In [19]:
print("=== NaN count per column in X_train ===")
nan_counts = X_train.isna().sum()
print(nan_counts[nan_counts > 0])

print(f"\nTotal rows in X_train: {len(X_train)}")
print(f"Rows with at least one NaN: {X_train.isna().any(axis=1).sum()}")

=== NaN count per column in X_train ===
visibility_prev_reading    4
dtype: int64

Total rows in X_train: 29360
Rows with at least one NaN: 4


In [21]:


constant_cols = [c for c in X_train.columns if X_train[c].nunique() <= 1]
print(f"Removing constant columns: {constant_cols}")

X_train_svm = X_train.drop(columns=constant_cols)
X_test_svm = X_test.drop(columns=constant_cols)

# Drop rows with NaN (only 4 rows — first reading per city has no previous value)
train_valid_idx = X_train_svm.dropna().index
X_train_svm = X_train_svm.loc[train_valid_idx]
y_train_svm = y_train.loc[train_valid_idx]
sample_weights_svm = sample_weights[X_train.index.isin(train_valid_idx)]

test_valid_idx = X_test_svm.dropna().index
X_test_svm = X_test_svm.loc[test_valid_idx]
y_test_svm = y_test.loc[test_valid_idx]

print(f"Training rows after dropping NaN: {len(X_train_svm)} (was {len(X_train)})")
print(f"Testing rows after dropping NaN: {len(X_test_svm)} (was {len(X_test)})")

print("\nScaling features for SVM...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_svm)
X_test_scaled = scaler.transform(X_test_svm)

print(f"Any NaN in scaled training data? {np.isnan(X_train_scaled).any()}")

print("Training SVM (this may take a few minutes)...")
start = time.time()

svm_model = SVR(kernel='rbf')
svm_model.fit(X_train_scaled, y_train_svm, sample_weight=sample_weights_svm)
svm_pred = svm_model.predict(X_test_scaled)

print(f"SVM trained in {time.time()-start:.1f} seconds")
results.append(evaluate_model("SVM", y_test_svm, svm_pred))


comparison_df = pd.DataFrame(results)
print("\n=== FINAL COMPARISON TABLE ===")
print(comparison_df.to_string(index=False))

Removing constant columns: ['snow']
Training rows after dropping NaN: 29356 (was 29360)
Testing rows after dropping NaN: 7340 (was 7340)

Scaling features for SVM...
Any NaN in scaled training data? False
Training SVM (this may take a few minutes)...
SVM trained in 208.0 seconds

=== SVM ===
R²: 0.6522 | MAE: 3.2245 | RMSE: 5.5007
Classification Accuracy: 71.4% | CRITICAL Recall: 57.6%

=== FINAL COMPARISON TABLE ===
        model       r2      mae     rmse  accuracy  critical_recall
      XGBoost 0.911361 1.547117 2.777077  0.778474         0.600000
Random Forest 0.870252 1.620480 3.359892  0.785286         0.526829
          SVM 0.652238 3.224482 5.500674  0.713896         0.575610


In [25]:
print(weather_df['windspeed'].describe())

count    37234.000000
mean         3.773428
std          4.754351
min          0.000000
25%          0.000000
50%          2.200000
75%          6.600000
max         63.100000
Name: windspeed, dtype: float64
